# IM-Benchmark-Lab - Backend benchmark trên Google Colab

Notebook này chỉ chạy backend thực nghiệm, không khởi động GUI. Nó chạy CELF, GA và IGA thật trên cùng đồ thị IC-WC, lưu checkpoint và xuất JSON trực tiếp vào Google Drive.

In [49]:
%%bash
set -Eeuo pipefail
apt-get update -qq
apt-get install -y -qq git cmake build-essential g++ python3-dev libgomp1
PROJECT_DIR=/content/influence-maximization-benchmark
REPO_URL=https://github.com/SLMR05-05/influence-maximization-benchmark.git
BRANCH=minh
if [ -d "$PROJECT_DIR/.git" ]; then
  git -C "$PROJECT_DIR" fetch origin "$BRANCH"
  git -C "$PROJECT_DIR" checkout "$BRANCH"
  git -C "$PROJECT_DIR" pull --ff-only origin "$BRANCH"
else
  git clone --branch "$BRANCH" "$REPO_URL" "$PROJECT_DIR"
fi
git config --global --add safe.directory "$PROJECT_DIR"
test -f "$PROJECT_DIR/src/im_research/experiments/colab_benchmark.py"
echo "Project: $PROJECT_DIR"
python --version
cmake --version | head -n 1
g++ --version | head -n 1

Your branch is up to date with 'origin/minh'.
Already up to date.
Project: /content/influence-maximization-benchmark
Python 3.13.16
cmake version 3.31.10
g++ (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
From https://github.com/SLMR05-05/influence-maximization-benchmark
 * branch            minh       -> FETCH_HEAD
Already on 'minh'
From https://github.com/SLMR05-05/influence-maximization-benchmark
 * branch            minh       -> FETCH_HEAD


In [50]:
%%bash
set -Eeuo pipefail
PROJECT_DIR=/content/influence-maximization-benchmark
python -m pip install --no-cache-dir -r "$PROJECT_DIR/requirements.txt"
cd "$PROJECT_DIR"
rm -rf build
cmake -S . -B build -DCMAKE_BUILD_TYPE=Release -DCMAKE_CXX_FLAGS="-O3 -fopenmp"
cmake --build build --parallel "$(nproc)"
python -m pip install --no-cache-dir --no-build-isolation -e .
echo "Backend build completed"

-- The CXX compiler identification is GNU 13.3.0
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Found OpenMP_CXX: -fopenmp (found version "4.5")
-- Found OpenMP: TRUE (found version "4.5")
-- OpenMP found! Enabling multi-threaded simulations.
-- Found PythonInterp: /usr/bin/python3 (found suitable version "3.13.16", minimum required is "3.9")
-- Found PythonLibs: /usr/lib/x86_64-linux-gnu/libpython3.13.so
-- Performing Test HAS_FLTO_AUTO
-- Performing Test HAS_FLTO_AUTO - Success
-- Found pybind11: /usr/local/lib/python3.13/dist-packages/pybind11/include (found version "3.1.0")
-- pybind11 found! Building Python extension module '_im_core'.
-- Configuring done (0.7s)
-- Generating done (0.0s)
-- Build files have been written to: /content/influence-maximization-benchmark/build
[ 12%] Building CXX object CMakeFiles/im_core_l

CMake Warning (dev) at /usr/local/lib/python3.13/dist-packages/pybind11/share/cmake/pybind11/FindPythonLibsNew.cmake:101 (message):
  Policy CMP0148 is not set: The FindPythonInterp and FindPythonLibs modules
  are removed.  Run "cmake --help-policy CMP0148" for policy details.  Use
  the cmake_policy command to set the policy and suppress this warning, or
  preferably upgrade to using FindPython, either by calling it explicitly
  before pybind11, or by setting PYBIND11_FINDPYTHON ON before pybind11.
Call Stack (most recent call first):
  /usr/local/lib/python3.13/dist-packages/pybind11/share/cmake/pybind11/pybind11Tools.cmake:44 (find_package)
  /usr/local/lib/python3.13/dist-packages/pybind11/share/cmake/pybind11/pybind11Common.cmake:243 (include)
  /usr/local/lib/python3.13/dist-packages/pybind11/share/cmake/pybind11/pybind11Config.cmake:257 (include)
  CMakeLists.txt:51 (find_package)
This warning is for project developers.  Use -Wno-dev to suppress it.

  /usr/local/lib/python3.13

In [51]:
from google.colab import drive
from pathlib import Path
import sys
import importlib

drive.mount('/content/drive')
PROJECT_DIR = Path('/content/influence-maximization-benchmark')
sys.path.insert(0, str(PROJECT_DIR / 'src'))
for module_name in list(sys.modules):
    if module_name == 'im_research' or module_name.startswith('im_research.'):
        del sys.modules[module_name]
importlib.invalidate_caches()

# Chỉ sửa các giá trị trong khối cấu hình này.
CONFIG = {
    'dataset_name': 'Karate Club',
    'dataset_path': None,  # Ví dụ: '/content/drive/MyDrive/IM-Benchmark-Lab/data/my_graph.txt'
    'directed': False,
    'diffusion_model': 'IC',
    'probability_model': 'WC',
    'p': 0.1,
    'k': 10,
    'mc_runs': 100,
    'validation_mc_runs': 1000,
    'independent_runs': 3,
    'algorithms': ['CELF', 'GA', 'IGA'],
    'population_size': 30,
    'generations': 50,
    'elite_size': 2,
    'crossover_rate': 0.8,
    'mutation_rate': 0.1,
    'tournament_size': 4,
    'random_seed': 42,
    'record_population_history': True,
    'record_mc_details': False,
    'record_failed_edges': False,
    'resume': True,
}

OUTPUT_DIR = Path('/content/drive/MyDrive/IM-Benchmark-Lab/Experiments') / CONFIG['dataset_name'].replace(' ', '_')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output root:', OUTPUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Output root: /content/drive/MyDrive/IM-Benchmark-Lab/Experiments/Karate_Club


In [52]:
from datetime import datetime
from im_research.experiments.colab_benchmark import run_benchmark

experiment_dir = OUTPUT_DIR / ('experiment_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
results = run_benchmark(
    **CONFIG,
    output_dir=experiment_dir,
)
print('Experiment completed:', results['experiment_id'])
print('Saved to:', experiment_dir)
print('Summary:')
for name, metrics in results['summary'].items():
    print(name, metrics['final_spread'])

[INFO] Dataset: Karate Club
[INFO] Graph: 34 nodes, 78 edges
[INFO] Diffusion: IC / WC
[INFO] Algorithms: CELF, GA, IGA
[INFO] Starting independent run 1/3 (seed=42)
[CELF] Starting search
[CELF] Completed: spread=26.3370, time=0.024s
[GA] Starting search
      [BaselineGA] Khởi tạo ngẫu nhiên quần thể N=30...
      [BaselineGA] Hoàn tất t=0 | Best Spread = 24.15 | Đa dạng Jaccard = 0.818
      [BaselineGA] Thế hệ   1/50 | Best Spread:  24.69 | Mean:  22.38 | Đa dạng: 0.713
      [BaselineGA] Thế hệ   5/50 | Best Spread:  25.93 | Mean:  24.41 | Đa dạng: 0.600
      [BaselineGA] Thế hệ  10/50 | Best Spread:  26.83 | Mean:  25.73 | Đa dạng: 0.396
      [BaselineGA] Thế hệ  15/50 | Best Spread:  27.18 | Mean:  26.01 | Đa dạng: 0.420
      [BaselineGA] Thế hệ  20/50 | Best Spread:  27.46 | Mean:  25.94 | Đa dạng: 0.314
      [BaselineGA] Thế hệ  25/50 | Best Spread:  27.50 | Mean:  26.07 | Đa dạng: 0.395
      [BaselineGA] Thế hệ  30/50 | Best Spread:  27.50 | Mean:  26.31 | Đa dạng: 0.363

In [53]:
import json
from pathlib import Path

manifest = json.loads((experiment_dir / 'manifest.json').read_text(encoding='utf-8'))
for filename in manifest['files']:
    path = experiment_dir / filename
    print(f'{filename:24s}', 'OK' if path.exists() else 'MISSING')

summary = json.loads((experiment_dir / 'summary.json').read_text(encoding='utf-8'))
for algorithm, metrics in summary.items():
    print(f"{algorithm}: mean={metrics['final_spread']['mean']:.4f}, std={metrics['final_spread']['std']:.4f}")

config.json              OK
graph_info.json          OK
runs.json                OK
summary.json             OK
comparison.json          OK
statistical_tests.json   OK
experiment.json          OK
logs.txt                 OK
checkpoint.json          OK
CELF: mean=26.4133, std=0.0832
GA: mean=26.8447, std=0.0136
IGA: mean=26.9730, std=0.1037


In [54]:
%%bash
set -Eeuo pipefail
cd /content/influence-maximization-benchmark
export PYTHONPATH="$PWD/src:${PYTHONPATH:-}"
python -m pytest tests/unit -q

..............................                                           [100%]
30 passed in 1.43s
